# OpenPlaque — Run Optimized Boundary + Plaque Characterization on UCLA Data

Fresh Colab workflow:

1. Mount Google Drive
2. Clone/update OpenPlaque from GitHub
3. Overlay this repository-additions ZIP if the new modules are not yet in GitHub
4. Install dependencies and configure nnU-Net
5. Load the optimized boundary parameters from the **current JSON format** (`final_parameters_selected_on_all_cases`)
6. Load `Full_DICOM.zip`, auto-detect LAD/RCA/LCX or use UCLA fallback series
7. Run nnU-Net segmentation
8. Apply optimized boundary refinement
9. Classify plaque voxels by HU into plaque types
10. Save CSV, NIfTI masks, overlays, and HTML reports

Research use only. Not clinically validated. Not for diagnosis or medical decision-making.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone/update OpenPlaque and optionally overlay repository additions from this ZIP.
# If these files are already merged into GitHub, no ZIP overlay is needed.

from pathlib import Path
import sys, shutil, zipfile, os

REPO_DIR = Path('/content/OpenPlaque')
SRC_DIR = REPO_DIR / 'src'
ZIP_CANDIDATES = [
    Path('/content/OpenPlaque_Repository_Additions_UCLA_Plaque_Characterization.zip'),
    Path('/content/drive/MyDrive/OpenPlaque/OpenPlaque_Repository_Additions_UCLA_Plaque_Characterization.zip'),
]
EXTRACT_DIR = Path('/content/openplaque_plaque_characterization_additions')

if not REPO_DIR.exists():
    !git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
else:
    !git -C /content/OpenPlaque pull

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is not None:
    print('Overlaying repository additions from:', zip_path)
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(EXTRACT_DIR)
    possible = [
        EXTRACT_DIR / 'src' / 'openplaque',
        EXTRACT_DIR / 'openplaque_repo_additions_v12' / 'src' / 'openplaque',
    ]
    overlay_src = next((p for p in possible if p.exists()), None)
    if overlay_src is None:
        raise FileNotFoundError('Could not find src/openplaque inside additions ZIP.')
    target_src = SRC_DIR / 'openplaque'
    target_src.mkdir(parents=True, exist_ok=True)
    for py in overlay_src.glob('*.py'):
        shutil.copy2(py, target_src / py.name)
        print('Overlayed', py.name)
else:
    print('No additions ZIP found. Assuming plaque characterization modules are already in GitHub.')

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import openplaque
print('Using OpenPlaque from:', openplaque.__file__)

In [ ]:
# Install requirements. Make sure Colab runtime type is GPU.
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q pandas scipy matplotlib SimpleITK pydicom

!nvidia-smi

In [ ]:
# Configure nnU-Net environment.
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('nnUNet_results:', os.environ['nnUNet_results'])

In [ ]:
# Extract nnU-Net model weights if needed.
from pathlib import Path
import zipfile

MODEL_ZIP = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
MODEL_TARGET = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if MODEL_TARGET.exists():
    print('Model already extracted:', MODEL_TARGET)
else:
    if not MODEL_ZIP.exists():
        raise FileNotFoundError(f'Missing model ZIP: {MODEL_ZIP}')
    with zipfile.ZipFile(MODEL_ZIP) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model to /content/nnUNet_results')

!find /content/nnUNet_results -maxdepth 4 | head -80

In [ ]:
# Input/output paths.
from pathlib import Path

STUDY_ZIP = Path('/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip')

BEST_PARAMETER_CANDIDATES = [
    Path('/content/drive/MyDrive/OpenPlaque/Boundary_Parameter_Tuning/best_boundary_parameters.json'),
    Path('/content/drive/MyDrive/OpenPlaque/Boundary_Tuning/best_boundary_parameters.json'),
    Path('/content/drive/MyDrive/OpenPlaque/best_boundary_parameters.json'),
]
BEST_PARAMETERS_JSON = next((p for p in BEST_PARAMETER_CANDIDATES if p.exists()), None)
if BEST_PARAMETERS_JSON is None:
    raise FileNotFoundError('Could not find best_boundary_parameters.json. Expected current format with final_parameters_selected_on_all_cases.')

OUTPUT_DIR = Path('/content/drive/MyDrive/OpenPlaque/UCLA_Plaque_Characterization')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Study:', STUDY_ZIP)
print('Best parameters:', BEST_PARAMETERS_JSON)
print('Output:', OUTPUT_DIR)

In [ ]:
# Show the optimized boundary parameters exactly as loaded from the current JSON format.
from openplaque.run_new_data import load_best_boundary_parameters

optimized_params = load_best_boundary_parameters(BEST_PARAMETERS_JSON)
print('Optimized boundary parameters')
print('-' * 60)
for k, v in optimized_params.items():
    print(f'{k}: {v}')

In [ ]:
# Plaque type thresholds. Modify here if you want different HU cutoffs.
from openplaque.plaque_characterization import DEFAULT_THRESHOLDS
import json

PLAQUE_TYPE_THRESHOLDS = DEFAULT_THRESHOLDS
print(json.dumps(PLAQUE_TYPE_THRESHOLDS, indent=2))

In [ ]:
# Run end-to-end processing on the UCLA data.
# This may take several minutes because nnU-Net inference runs for LAD/RCA/LCX.

from openplaque.run_characterization_new_data import process_new_study_with_characterization

results = process_new_study_with_characterization(
    study_zip=STUDY_ZIP,
    best_parameters_json=BEST_PARAMETERS_JSON,
    output_dir=OUTPUT_DIR,
    thresholds=PLAQUE_TYPE_THRESHOLDS,
    fallback_series={'RCA': 1035, 'LCX': 1039, 'LAD': 1043},
    save_masks=True,
    save_png_overlays=True,
    save_class_maps=True,
)

print('Done.')
print('Boundary HTML:', results['boundary_html'])
print('Characterization HTML:', results['characterization_html'])
print('Composition CSV:', results['composition_csv'])
print('Lesion CSV:', results['lesion_csv'])

In [ ]:
# Display per-vessel plaque composition in the notebook.
import pandas as pd

composition_df = pd.read_csv(results['composition_csv'])
lesion_df = pd.read_csv(results['lesion_csv'])

display(composition_df)
display(lesion_df.sort_values(['vessel', 'volume_mm3'], ascending=[True, False]).head(20))

In [ ]:
# Display generated output files.
!find /content/drive/MyDrive/OpenPlaque/UCLA_Plaque_Characterization -maxdepth 3 -type f | sort